# Скачивание аудио через Bot API (нужен только токен бота)

Этот способ НЕ требует api_id / api_hash и my.telegram.org — только
токен бота от @BotFather. Бот должен быть админом канала.

**Ограничение Telegram:** через Bot API можно скачивать файлы до 20 МБ.
Более крупные файлы этим способом не скачать.

**Как работает:** бот видит только новые сообщения. Пока работает
ячейка «Запуск», перешлите аудио в канал — бот их поймает и скачает
с названиями.

## 1. Токен бота
Вставьте токен и запустите.

In [ ]:
BOT_TOKEN = "ВСТАВЬТЕ_ТОКЕН_СЮДА"

## 2. Запуск приёмника
Запустите ячейку, затем перешлите аудио в канал со своего телефона.
Остановится сама после 90 секунд без новых файлов (или нажмите ⏹).

In [ ]:
import requests, json, re, time
from pathlib import Path

API = f'https://api.telegram.org/bot{BOT_TOKEN}'
FILEAPI = f'https://api.telegram.org/file/bot{BOT_TOKEN}'
OUT = Path('audio'); OUT.mkdir(exist_ok=True)
manifest_path = OUT / 'manifest.json'
manifest = json.loads(manifest_path.read_text('utf-8')) if manifest_path.exists() else []
used = {m['file'] for m in manifest}

def clean(s):
    s = re.sub(r'[\\/:*?"<>|\n\r\t]+', ' ', (s or '').strip())
    return (re.sub(r'\s+', ' ', s).strip()[:150]) or 'audio'

me = requests.get(f'{API}/getMe').json()
if not me.get('ok'):
    raise SystemExit('Неверный токен бота: ' + str(me))
print('Бот:', '@' + me['result']['username'])
print('Перешлите аудио в канал. Жду новые сообщения...')

def pick_audio(msg):
    if 'audio' in msg:
        a = msg['audio']
        title = a.get('title') or a.get('file_name')
        perf = a.get('performer')
        if perf and a.get('title'): title = f"{perf} - {a['title']}"
        return a, title, '.mp3'
    if 'voice' in msg:
        return msg['voice'], (msg.get('caption') or 'voice'), '.ogg'
    if 'document' in msg and str(msg['document'].get('mime_type','')).startswith('audio'):
        d = msg['document']
        return d, (d.get('file_name') or 'audio'), Path(d.get('file_name','x.mp3')).suffix or '.mp3'
    return None, None, None

def download(file_id, title, ext):
    info = requests.get(f'{API}/getFile', params={'file_id': file_id}).json()
    if not info.get('ok'):
        print('  пропуск (вероятно >20МБ):', info.get('description')); return False
    path = info['result']['file_path']
    name = clean(Path(title).stem if '.' in str(title) else title) + ext; i = 2
    while name in used or (OUT / name).exists():
        name = f'{clean(title)} ({i}){ext}'; i += 1
    used.add(name)
    print('↓', name)
    r = requests.get(f'{FILEAPI}/{path}')
    (OUT / name).write_bytes(r.content)
    manifest.append({'title': title, 'file': name})
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), 'utf-8')
    print('  всего:', len(manifest))
    return True

offset = None; last = time.time()
while True:
    r = requests.get(f'{API}/getUpdates', params={'offset': offset, 'timeout': 30,
                     'allowed_updates': json.dumps(['channel_post','message'])}, timeout=40).json()
    for upd in r.get('result', []):
        offset = upd['update_id'] + 1
        msg = upd.get('channel_post') or upd.get('message')
        if not msg: continue
        obj, title, ext = pick_audio(msg)
        if obj:
            if download(obj['file_id'], title, ext): last = time.time()
    if time.time() - last > 90 and manifest:
        print('90 секунд без новых файлов — стоп.'); break
print('Готово. Всего файлов:', len(manifest))

## 3. Скачать всё одним архивом

In [ ]:
import shutil
shutil.make_archive('audio_export', 'zip', 'audio')
from google.colab import files
files.download('audio_export.zip')